In [94]:
import pandas as pd
import keras
import tensorflow as tf 
import numpy as np
from sklearn.preprocessing import label_binarize
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

In [58]:
performance_dict ={'MODEL':['Baseline cnn','Deep cnn','Cnn batch normalization','Cnn Dropout','Mobile net','Efficentnet','Resnet','Densenet'],
'ACCURACY': [0.48,0.65,0.50,0.46,0.61,0.67,0.66,0.58],
'LOSS':[1.34,1.91,1.70,1.35,0.96,0.74,1.21,1.07]

}

In [59]:
df = pd.DataFrame(performance_dict)

In [60]:
df.to_csv(r'../data/selected_col/performance.csv')

In [61]:
df_test = pd.read_csv(r'../data/selected_col/model_test.csv')

df_test = df_test.drop(columns=['Unnamed: 0'])

In [62]:
df_test

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,dx_encode
0,HAM_0000627,ISIC_0034019,nv,consensus,50.0,unknown,unknown,../data/all_image/ISIC_0034019.jpg,5
1,HAM_0001761,ISIC_0026732,nv,follow_up,30.0,male,abdomen,../data/all_image/ISIC_0026732.jpg,5
2,HAM_0002726,ISIC_0025473,nv,consensus,5.0,male,foot,../data/all_image/ISIC_0025473.jpg,5
3,HAM_0003424,ISIC_0033552,nv,histo,45.0,male,back,../data/all_image/ISIC_0033552.jpg,5
4,HAM_0004081,ISIC_0031957,mel,histo,70.0,female,lower extremity,../data/all_image/ISIC_0031957.jpg,4
...,...,...,...,...,...,...,...,...,...
1498,HAM_0006808,ISIC_0027438,nv,follow_up,65.0,male,lower extremity,../data/all_image/ISIC_0027438.jpg,5
1499,HAM_0002425,ISIC_0032682,nv,consensus,20.0,male,back,../data/all_image/ISIC_0032682.jpg,5
1500,HAM_0003747,ISIC_0026582,nv,follow_up,50.0,male,trunk,../data/all_image/ISIC_0026582.jpg,5
1501,HAM_0002807,ISIC_0025050,nv,follow_up,45.0,male,trunk,../data/all_image/ISIC_0025050.jpg,5


In [63]:
basic_cnn_model = keras.models.load_model(r'../models/bmodel_cnn_RMS.keras')

In [64]:
def preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (224,224))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

In [65]:
def preprocess_test(image_path, label):
    image, label = preprocess_image(image_path, label)
    return image, label

In [66]:
test_dataset = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset = (
    test_dataset
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

In [67]:
predict_prob = basic_cnn_model.predict(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 22s 454ms/step


In [68]:
y_pred = np.argmax(predict_prob, axis=1)

In [69]:
y_true = []

for images, labels in test_dataset:
    y_true.extend(labels.numpy())

y_true = np.array(y_true)

In [70]:
evalution_dict ={}
evalution_dict['model'] =[]
evalution_dict['accuracy'] =[]
evalution_dict['precision'] = []
evalution_dict['recall'] = []
evalution_dict['roc-auc'] = []
evalution_dict['f1'] = []

In [71]:
'''baseline'''

'baseline'

In [72]:
accuracy = accuracy_score(y_true, y_pred)

print("Accuracy :", accuracy)

Accuracy : 0.6074517631403858


In [73]:
evalution_dict['model'].append('baseline cnn')
evalution_dict['accuracy'].append(accuracy)

In [74]:
precision = precision_score(
    y_true,
    y_pred,
    average="weighted"
)

print("Precision :", precision)

Precision : 0.6833055493420332


In [75]:
evalution_dict['precision'].append(precision)

In [76]:
recall = recall_score(
    y_true,
    y_pred,
    average="weighted"
)

print("Recall :", recall)

Recall : 0.6074517631403858


In [77]:
evalution_dict['recall'].append(recall)

In [78]:
f1 = f1_score(y_true,y_pred,average="weighted")

print("F1 Score :", f1)

F1 Score : 0.6152125021996068


In [79]:
evalution_dict['f1'].append(f1)

In [80]:


y_true_bin = label_binarize(
    y_true,
    classes=np.arange(7)
)

In [81]:
roc = roc_auc_score(y_true_bin,predict_prob,multi_class='ovr')
evalution_dict['roc-auc'].append(roc)
roc

0.8698757314746014

In [82]:
confusion_mat = confusion_matrix(y_true,y_pred)
confusion_mat

array([[  5,  11,  18,   2,   2,  11,   0],
       [  0,  39,  28,   3,   1,   6,   0],
       [  0,   8, 124,   0,   4,  29,   0],
       [  0,   9,   5,   2,   0,   1,   0],
       [  1,   8,  71,   1,  12,  71,   3],
       [  8,  16, 230,  11,  15, 722,   4],
       [  0,   6,   5,   1,   1,   0,   9]])

In [83]:
def calculation(y_true,y_pred,predict_prob):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true,y_pred,average="weighted")
    recall = recall_score(y_true,y_pred,average="weighted")
    f1 = f1_score(y_true,y_pred,average="weighted")
   

    y_true_bin = label_binarize(y_true,classes=np.arange(7))

    roc = roc_auc_score(y_true_bin,predict_prob,multi_class='ovr')

    return accuracy,precision,recall,f1,roc


In [84]:
def mat_app(model,accuracy,precision,recall,f1,roc):
    evalution_dict['model'].append(model)
    evalution_dict['accuracy'].append(accuracy)
    evalution_dict['precision'].append(precision)
    evalution_dict['recall'].append(recall)
    evalution_dict['f1'].append(f1)
    evalution_dict['roc-auc'].append(roc)

    return evalution_dict

In [85]:
deep_cnn_model = keras.models.load_model(r'../models/deep_model_adam.keras')

In [86]:
predict_prob = deep_cnn_model.predict(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 30s 619ms/step


In [87]:
y_pred = np.argmax(predict_prob, axis=1)

In [88]:
accuracy,precision,recall,f1,roc = calculation(y_true,y_pred,predict_prob)

In [89]:
accuracy,precision,recall,f1,roc

(0.5029940119760479,
 0.6914449947547051,
 0.5029940119760479,
 0.5572814312505384,
 0.8278161193490473)

In [90]:
mat_app('deep_cnn',accuracy,precision,recall,f1,roc)

{'model': ['baseline cnn', 'deep_cnn'],
 'accuracy': [0.6074517631403858, 0.5029940119760479],
 'precision': [0.6833055493420332, 0.6914449947547051],
 'recall': [0.6074517631403858, 0.5029940119760479],
 'roc-auc': [0.8698757314746014, 0.8278161193490473],
 'f1': [0.6152125021996068, 0.5572814312505384]}

In [95]:
import matplotlib.pyplot as plt

# Assuming 'history' is saved from model.fit()
plt.plot(basic_cnn_model.history['accuracy'], label='Training Accuracy')
plt.plot(basic_cnn_model.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.grid(alpha = 0.3)
plt.tight_layout()
plt.show()


AttributeError: 'Sequential' object has no attribute 'history'

In [97]:
from sklearn.metrics import auc, roc_curve

# Assuming y_test are true labels and preds are predicted probabilities
fpr, tpr, thresholds = roc_curve(y_true, predict_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(
    fpr, tpr, color='darkorange', label=f'ROC curve (area = {roc_auc:.2f})'
)
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc='lower right')
plt.show()


ValueError: multiclass format is not supported